# Long-Context RAG

Long-Context RAG explores how an LLM can work with a **large amount of
information within its context window**.

Traditional RAG retrieves a small number of relevant chunks, while
Long-Context RAG allows the LLM to process much more information at once.

The main goal is to understand:

- **Context windows** and their limitations
- **Large-context prompting**
- **Lost-in-the-Middle** problem
- **Traditional RAG vs Long-Context RAG**
- Context **selection and compression**
- Combining **retrieval with large context**

## Traditional RAG vs Long-Context RAG

| Traditional RAG                    | Long-Context RAG                            |
| ---------------------------------- | ------------------------------------------- |
| Retrieves a small number of chunks | Provides a larger amount of context         |
| Relies heavily on retrieval        | Can reduce reliance on aggressive retrieval |
| Sends a small context to the LLM   | Sends a large selected context              |
| Faster and more context-efficient  | Can process more information at once        |

### Traditional RAG

```text
Documents
    ↓
Chunking
    ↓
Embedding
    ↓
Vector Store
    ↓
User Query
    ↓
Retrieve Top-K
    ↓
Small Context
    ↓
LLM
    ↓
Answer

#Long-Context RAG

Documents
    ↓
Select / Organize Information
    ↓
Large Context
    ↓
LLM
    ↓
Answer
```


In [1]:
# Local Ollama LLM use garna
from langchain_ollama import ChatOllama

# Local Llama model load garne
llm = ChatOllama(model="llama3:latest", temperature=0)

print("LLM loaded successfully!")

LLM loaded successfully!


In [2]:
# Long context ko represent garne sample document create garne
document = """
Artificial Intelligence is a field of computer science concerned with
building systems that can perform tasks that normally require human
intelligence.

Machine learning is a subset of artificial intelligence where systems learn
patterns from data.

Deep learning is a subset of machine learning that uses neural networks with
multiple layers.

Retrieval-Augmented Generation combines information retrieval with language
generation. A RAG system retrieves relevant external information and provides
it to a language model as context.

Vector databases store numerical representations of information called
embeddings. These embeddings allow semantic similarity search.

Reranking improves retrieval by taking initially retrieved documents and
ordering them according to their relevance to the query.

Long-context processing allows language models to reason over a much larger
amount of information within a single context window.

The lost-in-the-middle problem occurs when relevant information appears in
the middle of a long context and the model may use that information less
effectively than information near the beginning or end.

Context compression attempts to reduce the amount of information provided to
the model while preserving the information necessary to answer the query.
"""

print(document)


Artificial Intelligence is a field of computer science concerned with
building systems that can perform tasks that normally require human
intelligence.

Machine learning is a subset of artificial intelligence where systems learn
patterns from data.

Deep learning is a subset of machine learning that uses neural networks with
multiple layers.

Retrieval-Augmented Generation combines information retrieval with language
generation. A RAG system retrieves relevant external information and provides
it to a language model as context.

Vector databases store numerical representations of information called
embeddings. These embeddings allow semantic similarity search.

Reranking improves retrieval by taking initially retrieved documents and
ordering them according to their relevance to the query.

Long-context processing allows language models to reason over a much larger
amount of information within a single context window.

The lost-in-the-middle problem occurs when relevant information app

In [3]:
# Entire document lai context ko rup ma LLM lai dine
query = "What is the relationship between RAG, vector databases, and reranking?"

prompt = f"""
Answer the question using only the provided context.

Context:
{document}

Question:
{query}

Answer:
"""

response = llm.invoke(prompt)

print(response.content)

Based on the provided context, there is no direct relationship mentioned between RAG, vector databases, and reranking. They are separate concepts:

* RAG (Retrieval-Augmented Generation) is a system that retrieves external information and provides it to a language model as context.
* Vector databases store numerical representations of information called embeddings, allowing semantic similarity search.
* Reranking is a technique that improves retrieval by reordering initially retrieved documents according to their relevance to the query.

There is no connection or interaction mentioned between these three concepts.


In [4]:
# Context ko beginning ma important information rakhne
beginning_context = """
IMPORTANT FACT:
The user's favorite programming language is Python.

""" + "\n".join(
    [f"Additional information {i}: This is unrelated information." for i in range(100)]
)

# Context ko middle ma important information rakhne
middle_information = "\n".join(
    [f"Additional information {i}: This is unrelated information." for i in range(50)]
)

middle_context = (
    "\n".join(
        [
            f"Additional information {i}: This is unrelated information."
            for i in range(50)
        ]
    )
    + "\n\nIMPORTANT FACT:\nThe user's favorite programming language is Python.\n\n"
    + "\n".join(
        [
            f"Additional information {i}: This is unrelated information."
            for i in range(50, 100)
        ]
    )
)

# Context ko end ma important information rakhne
end_context = (
    "\n".join(
        [
            f"Additional information {i}: This is unrelated information."
            for i in range(100)
        ]
    )
    + "\n\nIMPORTANT FACT:\nThe user's favorite programming language is Python."
)

print("Contexts created successfully!")

Contexts created successfully!


In [5]:
# Long context ko position based query test garne
def test_context(context):

    prompt = f"""
Use only the provided context to answer the question.

Context:
{context}

Question:
What is the user's favorite programming language?

Answer:
"""

    response = llm.invoke(prompt)

    return response.content

In [6]:
# Beginning position test
beginning_response = test_context(beginning_context)

print("Beginning:")
print(beginning_response)

Beginning:
The user's favorite programming language is Python.


In [7]:
# Middle position test
middle_response = test_context(middle_context)

print("Middle:")
print(middle_response)

Middle:
According to the context, the user's favorite programming language is Python.


In [8]:
# End position test
end_response = test_context(end_context)

print("End:")
print(end_response)

End:
According to the provided context, the user's favorite programming language is Python.


In [9]:
# Large context lai smaller chunks ma divide garne
def create_context_chunks(text, chunk_size=500):

    words = text.split()

    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i : i + chunk_size])

        chunks.append(chunk)

    return chunks


# Document chunks create garne
chunks = create_context_chunks(document, chunk_size=50)

print(f"Total chunks: {len(chunks)}")

Total chunks: 4


In [10]:
# Relevant information identify garna simple keyword-based selection
# This is a learning demonstration before using semantic retrieval.
def select_relevant_context(chunks, keywords):

    selected_chunks = []

    for chunk in chunks:

        if any(keyword.lower() in chunk.lower() for keyword in keywords):
            selected_chunks.append(chunk)

    return selected_chunks


selected_chunks = select_relevant_context(chunks, ["RAG", "vector", "reranking"])

print(f"Selected chunks: {len(selected_chunks)}")

for chunk in selected_chunks:
    print("\n", chunk)

Selected chunks: 1

 Retrieval-Augmented Generation combines information retrieval with language generation. A RAG system retrieves relevant external information and provides it to a language model as context. Vector databases store numerical representations of information called embeddings. These embeddings allow semantic similarity search. Reranking improves retrieval by taking initially retrieved documents and ordering them


In [11]:
# Selected context lai LLM bata compress garne
def compress_context(context, query):

    prompt = f"""
Extract only the information from the context that is necessary to answer
the question.

Do not add new information.
Do not answer the question.
Only return the relevant information.

Context:
{context}

Question:
{query}

Relevant information:
"""

    response = llm.invoke(prompt)

    return response.content

In [12]:
# Selected chunks combine garne
large_context = "\n\n".join(selected_chunks)

query = "How does RAG use vector databases and reranking?"

compressed_context = compress_context(large_context, query)

print(compressed_context)

Vector databases store numerical representations of information called embeddings. These embeddings allow semantic similarity search.


In [13]:
# Compressed context use garera final answer generate garne
prompt = f"""
Answer the question using only the provided context.

Context:
{compressed_context}

Question:
{query}

Answer:
"""

response = llm.invoke(prompt)

print(response.content)

According to the provided context, RAG (Re-ranking with Adversarial Generation) uses vector databases to store numerical representations of information called embeddings, which enables semantic similarity search.
